# 2.1.2 CLP — 그래픽 해법 예시 (Graphic Solution Example)

## 소개 (Introduction)

이 노트북에서는 두 변수를 갖는 CLP 문제를 <strong>그래픽 해법(Graphic Method)</strong>으로 푸는 방법을 분석합니다.

먼저 문제를 CLP 모델로 수식화한 뒤, Python의 `matplotlib` 라이브러리로 의사결정 공간을 시각화하여 최적해를 찾습니다.

> **참고:** 그래픽 해법은 실용적인 풀이법이라기보다, CLP에 대한 직관적 이해를 얻기 위한 **교육적 도구**입니다. 이후 다룰 <strong>심플렉스 알고리즘(Simplex Algorithm)</strong>은 이 방법의 수학적 확장입니다.

예시 문제로 간단한 <strong>블렌딩 문제(Blending Problem)</strong>를 사용합니다.

## 문제 정의 (Problem Definition)

두 가지 재료 $M_1$, $M_2$를 혼합하여 식품 제품을 만듭니다.

**비용:**
- $M_1$ 그램당 45 유로센트
- $M_2$ 그램당 12 유로센트

**영양 성분 (재료 100g당):**

| 재료 | 불포화지방(Non-saturated Fat) | 단백질(Protein) |
|---|---|---|
| $M_1$ | 3% | 80% |
| $M_2$ | 3% | 20% |

**품질 제약:**
- 불포화지방 $\leq$ 0.5%
- 단백질 $\geq$ 8%
- $M_2$ 사용량 $\geq$ 2% (레시피 최소 요건)

**> 제품 100그램당 $M_1$과 $M_2$를 각각 몇 그램씩 사용해야 비용이 최소화되는가?**

## 모델 (Model)

**결정 변수:**
- $x_1$: 제품 100g당 $M_1$의 사용량 (그램)
- $x_2$: 제품 100g당 $M_2$의 사용량 (그램)

**목적 함수 (비용 최소화):**

$$\min Z = 45x_1 + 12x_2$$

**제약 조건:**

$$x_1 > 0$$

$$x_2 \geq 2$$

$$x_1 + x_2 \leq 100$$

$$3x_1 + 3x_2 \leq 50$$

$$8x_1 + 2x_2 \geq 80$$

처음 세 제약은 <strong>물리적 제약(Physical Constraints)</strong>입니다. 나머지 두 제약은 영양 요건에서 도출됩니다:
- $3x_1 + 3x_2 \leq 50$: 불포화지방 $\leq$ 0.5% (제품 100g 기준)
- $8x_1 + 2x_2 \geq 80$: 단백질 $\geq$ 8% (제품 100g 기준)

## 풀이 개요 (Solution Approach)

**사용 라이브러리:**
- `numpy.linspace(start, stop, n)`: 의사결정 변수 $x_1$의 축을 `n`개 등간격으로 생성
- `numpy.maximum(a, b)` / `numpy.minimum(a, b)`: 배열 원소별 최댓값/최솟값 — 가능 영역 경계선 계산에 사용
- `matplotlib.pyplot.fill_between(x, y1, y2)`: 두 선 사이 영역을 채워 <strong>가능 영역(Feasibility Region)</strong> 시각화

**그래픽 해법 절차:**
1. 제약 조건을 하나씩 추가하며 가능 영역을 좁혀 나감
2. 최종 가능 영역의 <strong>꼭짓점(Corner Points, Vertices)</strong>을 구함
3. 각 꼭짓점에서 목적 함수 $Z$ 값을 계산
4. 최솟값을 주는 꼭짓점이 최적해

> **원리:** 선형계획법에서 최적해는 항상 가능 영역의 꼭짓점에 존재합니다.

## 가능 영역 (Feasibility Region)

물리적 제약만 고려한 초기 가능 영역입니다:

$$x_1 > 0, \quad x_2 > 0, \quad x_1 + x_2 \leq 100$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

x = np.linspace(0, 100, 2000)

y1 = 100 - x   # x1 + x2 <= 100
y2 = x * 0     # x2 > 0

plt.plot(x, y1, label=r'$x_{1}+x_{2} \leq 100$')
plt.xlim((0, 100))
plt.ylim((0, 100))
plt.xlabel(r'$x_{1}$')
plt.ylabel(r'$x_{2}$')
plt.fill_between(x, y1, y2, where=y1 > y2, color='grey', alpha=0.5)
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
plt.title('물리적 제약만 적용한 가능 영역')

### 제약 추가 1단계 — $x_2 \geq 2$ 적용

$M_2$ 최소 사용량 제약을 추가합니다:

$$x_1 > 0, \quad x_2 \geq 2, \quad x_1 + x_2 \leq 100$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

x = np.linspace(0, 100, 2000)

y1 = 100 - x     # x1 + x2 <= 100
y2 = x * 0 + 2   # x2 >= 2

plt.plot(x, y1, label=r'$x_{1}+x_{2} \leq 100$')
plt.plot(x, y2, label=r'$x_{2} \geq 2$')
plt.xlim((0, 100))
plt.ylim((0, 100))
plt.xlabel(r'$x_{1}$')
plt.ylabel(r'$x_{2}$')
plt.fill_between(x, y1, y2, where=y1 > y2, color='grey', alpha=0.5)
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
plt.title('$x_2 \geq 2$ 제약 추가 후 가능 영역')

### 제약 추가 2단계 — 불포화지방 제약 $3x_1 + 3x_2 \leq 50$ 적용

$$x_1 > 0, \quad x_2 \geq 2, \quad x_1 + x_2 \leq 100, \quad 3x_1 + 3x_2 \leq 50$$

불포화지방 제약이 물리적 제약 $x_1 + x_2 \leq 100$보다 더 강하게 작용하여, 물리적 제약이 사실상 가능 영역에 영향을 미치지 않게 됩니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

x = np.linspace(0, 100, 2000)

y1 = 100 - x          # x1 + x2 <= 100
y2 = x * 0 + 2        # x2 >= 2
y3 = (50 - 3*x) / 3.0 # 3x1 + 3x2 <= 50

plt.plot(x, y1, label=r'$x_{1}+x_{2} \leq 100$')
plt.plot(x, y2, label=r'$x_{2} \geq 2$')
plt.plot(x, y3, label=r'$3x_{1}+3x_{2} \leq 50$')
plt.xlim((0, 100))
plt.ylim((0, 100))
plt.xlabel(r'$x_{1}$')
plt.ylabel(r'$x_{2}$')
plt.fill_between(x, y3, y2, where=y3 > y2, color='grey', alpha=0.5)
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
plt.title('불포화지방 제약 추가 후 가능 영역')

### 제약 추가 3단계 — 단백질 제약 $8x_1 + 2x_2 \geq 80$ 적용 (최종 가능 영역)

모든 제약을 적용합니다:

$$x_1 > 0, \quad x_2 \geq 2, \quad x_1 + x_2 \leq 100$$

$$3x_1 + 3x_2 \leq 50, \quad 8x_1 + 2x_2 \geq 80$$

단백질 요건 제약이 추가됨으로써 가능 영역이 크게 좁혀집니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

x = np.linspace(0, 100, 2000)

y1 = 100 - x           # x1 + x2 <= 100
y2 = x * 0 + 2         # x2 >= 2
y3 = (50 - 3*x) / 3.0  # 3x1 + 3x2 <= 50
y4 = (80 - 8*x) / 2.0  # 8x1 + 2x2 >= 80

plt.plot(x, y1, label=r'$x_{1}+x_{2} \leq 100$')
plt.plot(x, y2, label=r'$x_{2} \geq 2$')
plt.plot(x, y3, label=r'$3x_{1}+3x_{2} \leq 50$')
plt.plot(x, y4, label=r'$8x_{1}+2x_{2} \geq 80$')

plt.xlim((0, 20))
plt.ylim((0, 15))
plt.xlabel(r'$x_{1}$')
plt.ylabel(r'$x_{2}$')

# 가능 영역: y4와 y3 중 최댓값 ~ y2와 y4 중 최댓값 사이
y5 = np.maximum(y4, y3)
y6 = np.maximum(y2, y4)
plt.fill_between(x, y5, y6, where=y5 > y6, color='grey', alpha=0.5)

plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
plt.title('최종 가능 영역 (회색)')

## 꼭짓점 분석 (Corner Point Analysis)

**원리:** 선형계획법의 최적해(최솟값 또는 최댓값)는 항상 가능 영역의 꼭짓점에 존재합니다.

최종 가능 영역의 꼭짓점은 제약 경계선들의 교점입니다:

- 꼭짓점 A: $3x_1 + 3x_2 = 50$ ∩ $8x_1 + 2x_2 = 80$
- 꼭짓점 B: $8x_1 + 2x_2 = 80$ ∩ $x_2 = 2$
- 꼭짓점 C: $x_2 = 2$ ∩ $3x_1 + 3x_2 = 50$

목적 함수 $Z = 45x_1 + 12x_2$를 각 꼭짓점에서 계산합니다.

### 꼭짓점 좌표 계산

**꼭짓점 A** — $3x_1 + 3x_2 = 50$ 과 $8x_1 + 2x_2 = 80$ 의 교점:

$$\frac{50 - 3x_1}{3} = \frac{80 - 8x_1}{2}$$

$$x_1 = 7.78, \quad x_2 = 8.86, \quad Z = 45(7.78) + 12(8.86) = 456.42$$

---

**꼭짓점 B** — $8x_1 + 2x_2 = 80$ 과 $x_2 = 2$ 의 교점:

$$\frac{80 - 8x_1}{2} = 2 \implies x_1 = 9.5$$

$$x_1 = 9.5, \quad x_2 = 2, \quad Z = 45(9.5) + 12(2) = 451.5$$

---

**꼭짓점 C** — $x_2 = 2$ 과 $3x_1 + 3x_2 = 50$ 의 교점:

$$2 = \frac{50 - 3x_1}{3} \implies x_1 = 14.67$$

$$x_1 = 14.67, \quad x_2 = 2, \quad Z = 45(14.67) + 12(2) = 684$$

---

**결론:** $Z$가 최소인 꼭짓점은 **꼭짓점 B** → $x_1 = 9.5$, $x_2 = 2$, $Z_{min} = 451.5$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

x = np.linspace(0, 100, 2000)

y1 = 100 - x
y2 = x * 0 + 2
y3 = (50 - 3*x) / 3.0
y4 = (80 - 8*x) / 2.0
y_opt = (451.5 - 45*x) / 12  # 목적 함수 등고선 Z = 451.5

plt.plot(x, y1, label=r'$x_{1}+x_{2} \leq 100$')
plt.plot(x, y2, label=r'$x_{2} \geq 2$')
plt.plot(x, y3, label=r'$3x_{1}+3x_{2} \leq 50$')
plt.plot(x, y4, label=r'$8x_{1}+2x_{2} \geq 80$')
plt.plot(x, y_opt, label=r'$Z=451.5$ (최적 등고선)', linestyle='--', color='red')

plt.xlim((0, 20))
plt.ylim((0, 15))
plt.xlabel(r'$x_{1}$')
plt.ylabel(r'$x_{2}$')

y5 = np.maximum(y4, y3)
y6 = np.maximum(y2, y4)
plt.fill_between(x, y5, y6, where=y5 > y6, color='grey', alpha=0.5)

# 최적해 점 표시
plt.scatter([9.5], [2], color='red', zorder=5)
plt.annotate(r'최적해 $(9.5,\ 2)$', xy=(9.5, 2), xytext=(11, 4),
             arrowprops=dict(arrowstyle='->', color='red'), color='red')

plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
plt.title('최적해 시각화')

## 결론 (Conclusion)

그래픽 해법으로 최솟값 $Z = 451.5$를 구했습니다:

$$x_1 = 9.5 \text{ g}, \quad x_2 = 2 \text{ g} \quad \Rightarrow \quad Z_{min} = 451.5 \text{ 유로센트}$$

**한계:** 그래픽 해법은 변수가 2~3개인 소규모 문제에만 적용 가능합니다.
변수와 제약 수가 늘어날수록 시각화가 불가능해지고, 꼭짓점 개수가 기하급수적으로 증가합니다.

이러한 한계를 극복한 방법이 바로 <strong>심플렉스 알고리즘(Simplex Algorithm)</strong>입니다 — 이후 챕터에서 다룹니다.